In [1]:
import cv2
import os
import numpy as np

In [2]:
input_folder = "data/train/cracked"

In [3]:
output_folder = "data/train/cracked_aug"

In [13]:
import os
import shutil

base = r"C:\Users\Shreyash\FutureAI\Assignment2"

cracked_dst = os.path.join(base, "data/train/cracked")

for folder in os.listdir(base):
    folder_path = os.path.join(base, folder)

    if os.path.isdir(folder_path) and folder not in ["data", "None"]:
        for img in os.listdir(folder_path):
            shutil.move(os.path.join(folder_path, img), cracked_dst)

print("All defect images moved!")

All defect images moved!


In [15]:
import os

print(os.listdir(r"C:\Users\Shreyash\FutureAI\Assignment2\data\train"))

['cracked', 'healthy']


In [16]:
print("Healthy:", len(os.listdir(r"C:\Users\Shreyash\FutureAI\Assignment2\data\train\healthy")))
print("Cracked:", len(os.listdir(r"C:\Users\Shreyash\FutureAI\Assignment2\data\train\cracked")))

Healthy: 101
Cracked: 110


In [17]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

BASE = r"C:\Users\Shreyash\FutureAI\Assignment2"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_data = datasets.ImageFolder(
    root=BASE + "/data/train",
    transform=transform
)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

print(train_data.classes)

['cracked', 'healthy']


In [18]:
train_data.classes

['cracked', 'healthy']

In [19]:
import torch
import torch.nn as nn
from torchvision import models

model = models.resnet50(pretrained=True)

C:\Users\Shreyash\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Shreyash\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\Shreyash/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|█████████████████████████████████| 97.8M/97.8M [00:26<00:00, 3.90MB/s]


In [20]:
for param in model.parameters():
    param.requires_grad = False

In [21]:
model.fc = nn.Linear(model.fc.in_features, 2)

In [22]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.fc.parameters(),  # only train last layer
    lr=0.001
)

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [24]:
for epoch in range(5):   # 5 epochs enough for assignment
    model.train()

    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 10.1345
Epoch 2, Loss: 6.6955
Epoch 3, Loss: 6.1267
Epoch 4, Loss: 5.2776
Epoch 5, Loss: 5.0141


In [25]:
from PIL import Image
import torch

# Set model to evaluation mode
model.eval()

# Define same transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Class names (VERY IMPORTANT: same order as training)
class_names = train_data.classes
print("Classes:", class_names)

def predict_image(image_path):
    img = Image.open(image_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]

Classes: ['cracked', 'healthy']


In [27]:
import os

test_folder = r"C:\Users\Shreyash\FutureAI\Assignment2\data\test"

for img_name in os.listdir(test_folder):
    
    # Skip hidden/system files
    if img_name.startswith("."):
        continue

    path = os.path.join(test_folder, img_name)

    # Ensure it's a file (not folder)
    if not os.path.isfile(path):
        continue

    result = predict_image(path)
    print(f"{img_name} → {result}")

img1.jpg → cracked
img2.jpg → cracked
